# KernelForge — GPU session

**Settings → Accelerator → GPU T4** and **Internet → On**, then *Run All*.

Produces the GPU-verified dataset (`data/verified/dataset.jsonl`), the per-category pass/speedup report, and a baseline eval on every held-out prompt (set `EVAL_LIMIT` to cap it for a quicker run). Set `FINETUNE = True` to also train the QLoRA adapter and evaluate it on the same held-out ops (adds about an hour).

Everything lands in `/kaggle/working/kernelforge_outputs.zip` (Output panel → download). The zip is refreshed **after every stage** (and every 10 minutes during the dataset stage), so if a later stage fails or the session dies, the zip still has everything finished so far. `stage_status.json` inside it says which stages passed, failed or were skipped, and why.

**Resuming a dataset run that died:** upload the old `kernelforge_outputs.zip` as a Kaggle dataset, add it to this notebook, and set `RESUME_FROM` to its path under `/kaggle/input/`. Entries already verified are skipped.

In [ ]:
FINETUNE = False      # True: + QLoRA fine-tune and fine-tuned eval
EVAL_LIMIT = None     # None = every held-out prompt (100 with the default split); an int caps it, round-robin across categories
BRANCH = "main"
BUDGET_HOURS = 11     # stages that can't finish inside this are skipped cleanly; keep it below Kaggle's session limit
RESUME_FROM = ""      # e.g. "/kaggle/input/kf-prev/kernelforge_outputs.zip" to continue an earlier dataset run

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!rm -rf KernelForge && git clone -q --depth 1 -b {BRANCH} https://github.com/Arsalan-05/KernelForge.git
%cd KernelForge
!pip install -q -r requirements-demo.txt
!python -c "import torch, triton; print('torch', torch.__version__, '| triton', triton.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
flags = f"--budget-hours {BUDGET_HOURS}"
flags += f" --eval-limit {EVAL_LIMIT}" if EVAL_LIMIT else ""
flags += " --finetune" if FINETUNE else ""
flags += f" --resume-from {RESUME_FROM}" if RESUME_FROM else ""
# gpu_session.py catches stage failures itself and zips after every stage;
# a non-zero exit only means at least one stage failed (see the summary below).
!python scripts/gpu_session.py {flags}

In [ ]:
import json, os
path = "/kaggle/working/kernelforge_outputs/session.json"
if not os.path.exists(path):
    print("No session.json: gpu_session.py did not get far enough to write one (see its output above).")
else:
    s = json.load(open(path))
    print(json.dumps(s.get("env", {}), indent=2))
    for stage, r in s.get("stages", {}).items():
        print(f"{stage:<10} {r['status']:<8} {r.get('seconds', 0):>8}s  {r.get('error') or ''}")
        if "prompts_evaluated" in r:
            print(f"{'':<10} {r['prompts_evaluated']} of {r['held_out_total']} held-out prompts "
                  f"({'full set' if r['full_held_out_set'] else 'NOT the full set'})")
    rep = s.get("dataset_report", {})
    if rep.get("partial"):
        print(f"\nDataset run is PARTIAL: {len(rep['run_stats']['not_run'])} entries not run (resume to finish).")
    for cat, r in rep.get("per_category", {}).items():
        print(f"{cat:<18} {r}")

In [ ]:
# Last resort: if gpu_session.py crashed before writing its zip, zip whatever exists.
import os, shutil
out = "/kaggle/working/kernelforge_outputs"
if os.path.exists(out + ".zip"):
    print("Download:", out + ".zip")
else:
    os.makedirs(out, exist_ok=True)
    for rel in ["data/verified", "data/raw"]:
        if os.path.isdir(rel):
            shutil.copytree(rel, os.path.join(out, rel), dirs_exist_ok=True)
    print("Fallback zip:", shutil.make_archive(out, "zip", root_dir=out))